In [6]:
# ============================================================
# REPOSITORY SETUP — YOUR GitHub repository
# ============================================================

from pathlib import Path
import shutil
import urllib.request
import zipfile

ZIP_URL = "https://codeload.github.com/Imvixh/flyrank-ml-internship/zip/refs/heads/main"

ROOT = Path("/content/flyrank-ml-internship")
ZIP_PATH = Path("/content/flyrank-ml-internship-main.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("Downloading YOUR GitHub repository...")
urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)

print("Extracting repository...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall("/content")

EXTRACTED = Path("/content/flyrank-ml-internship-main")

if ROOT.exists():
    shutil.rmtree(ROOT)

EXTRACTED.rename(ROOT)
ZIP_PATH.unlink(missing_ok=True)

DATA = ROOT / "data/raw/content_refresh_anonymized.csv"

print("\n" + "=" * 55)
print("REPOSITORY CHECK")
print("=" * 55)

print("Repository exists:", ROOT.exists())
print("Dataset exists:", DATA.exists())
print("Repository:", ROOT)
print("Dataset:", DATA)

if not ROOT.exists():
    raise FileNotFoundError("Repository could not be prepared.")

if not DATA.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA}")

print("\n✓ YOUR repository ready")
print("✓ Dataset ready")

Extracting repository...

REPOSITORY CHECK
Repository exists: True
Dataset exists: True
Repository: /content/flyrank-ml-internship
Dataset: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

✓ YOUR repository ready
✓ Dataset ready


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

The baseline assigns a higher refresh priority to pages showing clear observed signs of declining search performance. The score combines the direction and size of the observed trend with supporting signals such as search demand, visibility, CTR, and ranking position.

The rule is intentionally transparent: it is a hand-written baseline that can be inspected and compared against the ML model later.

### Reason codes

- `declining_with_demand` — the page shows a downward trend while having measurable search demand.
- `low_ctr_visible_page` — the page has relatively weak CTR despite having visible search position.
- `ranking_opportunity` — the page has a visible ranking position where improvement may be worth human review.
- `stale_content` — the page has been available for a meaningful period or has not been updated recently.
- `low_visibility` — the page has limited observed search visibility.
- `insufficient_evidence` — available signals are too weak or incomplete for a confident recommendation.

The baseline is a prioritization rule, not a causal claim that refreshing a page will improve performance.

In [7]:
# ============================================================
# SECTION 1 — BASELINE RULE + REASON CODES
# ============================================================

import numpy as np
import pandas as pd

df = pd.read_csv(DATA)

def build_reason_codes(row):
    reasons = []

    trend = str(row.get("trend_direction", "")).lower()
    trend_pct = row.get("trend_pct", np.nan)
    search_volume = row.get("search_volume", np.nan)
    ctr = row.get("ctr", np.nan)
    position = row.get("avg_position", np.nan)

    if trend == "down":
        reasons.append("declining_with_demand")

    if (
        pd.notna(ctr)
        and pd.notna(position)
        and position <= 10
        and ctr < 1
    ):
        reasons.append("low_ctr_visible_page")

    if pd.notna(position) and position <= 20:
        reasons.append("ranking_opportunity")

    if (
        pd.notna(row.get("content_age_days", np.nan))
        and row["content_age_days"] >= 365
    ):
        reasons.append("stale_content")

    if (
        pd.notna(search_volume)
        and search_volume < 10
    ):
        reasons.append("low_visibility")

    if not reasons:
        reasons.append("insufficient_evidence")

    return "|".join(reasons)


df["baseline_reason_codes"] = df.apply(build_reason_codes, axis=1)

print("Rows:", len(df))
print("Reason codes created:", df["baseline_reason_codes"].notna().sum())

print("\nExample reason codes:")
display(
    df[
        ["content_id", "trend_direction", "trend_pct",
         "search_volume", "ctr", "avg_position",
         "baseline_reason_codes"]
    ].head(10)
)

Rows: 30000
Reason codes created: 30000

Example reason codes:


,content_id,trend_direction,trend_pct,search_volume,ctr,avg_position,baseline_reason_codes
0,content_304f48230142,down,-41.4,10.0,0.76,10.6,declining_with_demand|ranking_opportunity
1,content_a1fb4e703a9e,down,-57.7,90.0,0.05,20.3,declining_with_demand|stale_content
2,content_9aa793d4d895,down,-60.9,0.0,0.09,36.5,declining_with_demand|low_visibility
3,content_331d6c4de07b,stable,-13.8,10.0,0.49,6.2,low_ctr_visible_page|ranking_opportunity|stale...
4,content_d99b7a2d90ca,down,-34.7,0.0,0.13,44.0,declining_with_demand|low_visibility
5,content_d4084a4bc775,down,-38.9,720.0,0.03,8.5,declining_with_demand|low_ctr_visible_page|ran...
6,content_9a34b442b552,down,-92.3,0.0,0.00,7.0,declining_with_demand|low_ctr_visible_page|ran...
7,content_a63219c6e95a,stable,0.6,590.0,0.06,21.2,stale_content
8,content_5e6c160719bc,down,-58.8,0.0,0.09,46.0,declining_with_demand|low_visibility
9,content_c27558df2b0c,down,-29.2,0.0,0.16,4.9,declining_with_demand|low_ctr_visible_page|ran...


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring

The baseline score is a transparent combination of observed decline and supporting opportunity signals. Pages with a downward trend receive the strongest priority, with additional points for meaningful search demand, visible ranking position, and weak CTR. The resulting score is used only to rank pages for human review.

The output is written to `work/outputs/baseline_action_score.csv`.

In [8]:
# ============================================================
# SECTION 2 — BUILD RANKED BASELINE QUEUE
# ============================================================

import numpy as np
import pandas as pd

# Start with a transparent score of zero
df["baseline_score"] = 0.0

# Strongest signal: observed downward trend
df.loc[
    df["trend_direction"].astype(str).str.lower().eq("down"),
    "baseline_score"
] += 50

# Larger negative trend gets additional priority
trend_pct = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)

df["baseline_score"] += (
    (-trend_pct).clip(lower=0, upper=100) * 0.20
)

# Search demand signal
search_volume = pd.to_numeric(
    df["search_volume"], errors="coerce"
).fillna(0)

df["baseline_score"] += (
    np.log1p(search_volume).clip(upper=10) * 2
)

# Visible ranking opportunity
position = pd.to_numeric(
    df["avg_position"], errors="coerce"
)

df.loc[position <= 10, "baseline_score"] += 10
df.loc[(position > 10) & (position <= 20), "baseline_score"] += 5

# CTR opportunity
ctr = pd.to_numeric(
    df["ctr"], errors="coerce"
)

df.loc[
    (position <= 10) & (ctr < 1),
    "baseline_score"
] += 10

# Older content gets a small review-priority bonus
age = pd.to_numeric(
    df["content_age_days"], errors="coerce"
)

df.loc[age >= 365, "baseline_score"] += 5

# Rank all pages
queue = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["baseline_rank"] = np.arange(1, len(queue) + 1)

# Select useful output columns
output_columns = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_score",
    "baseline_reason_codes",
    "trend_direction",
    "trend_pct",
    "search_volume",
    "ctr",
    "avg_position",
    "content_age_days",
    "content_type",
    "main_intent",
]

output_columns = [
    c for c in output_columns
    if c in queue.columns
]

baseline_output = queue[output_columns].copy()

# Write required CSV
OUTPUT_DIR = ROOT / "work/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"

baseline_output.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Rows ranked:", len(baseline_output))
print("Output:", OUTPUT_PATH)
print("\nTop 20 baseline recommendations:")

display(baseline_output.head(20))

Rows ranked: 30000
Output: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv

Top 20 baseline recommendations:


,baseline_rank,content_id,client_id,baseline_score,baseline_reason_codes,trend_direction,trend_pct,search_volume,ctr,avg_position,content_age_days,content_type,main_intent
0,1,content_e8fc703f7ef0,client_3fdba35f04,112.700782,declining_with_demand|low_ctr_visible_page|ran...,down,-96.5,9900.0,0.00,9.5,463,keyword article,informational
1,2,content_548f0bf562a9,client_3fdba35f04,110.800782,declining_with_demand|low_ctr_visible_page|ran...,down,-87.0,9900.0,0.05,3.2,463,keyword article,informational
2,3,content_b933ddb19b17,client_19581e27de,109.677934,declining_with_demand|low_ctr_visible_page|ran...,down,-91.5,3600.0,0.00,5.1,466,keyword article,commercial
3,4,content_71b773ffcf59,client_4ec9599fc2,109.341777,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,1300.0,0.97,7.9,445,keyword article,informational
4,5,content_5b5e85993c2b,client_9400f1b21c,109.341777,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,1300.0,0.00,4.3,557,keyword article,informational
5,6,content_c3ccadb77bc4,client_19581e27de,108.396767,declining_with_demand|low_ctr_visible_page|ran...,down,-93.2,1600.0,0.00,2.4,482,keyword article,commercial
6,7,content_a7d5a0c2535b,client_19581e27de,108.200271,declining_with_demand|low_ctr_visible_page|ran...,down,-90.5,1900.0,0.00,2.8,417,keyword article,informational
7,8,content_e5a8021fc0d9,client_e629fa6598,108.161278,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,720.0,0.00,7.8,460,keyword article,informational
8,9,content_608540486d95,client_8722616204,107.999486,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,8100.0,0.00,7.0,228,keyword article,informational
9,10,content_9357e2dc8d36,client_e629fa6598,107.763632,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,590.0,0.00,9.4,460,keyword article,informational


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The top 20 pages are treated as a human-review queue rather than automatic refresh decisions. Each recommendation should be checked against the observed trend, search demand, CTR, ranking position, content age, and data completeness.

**Action:** Review the page for a possible refresh, with higher priority given to pages showing a downward trend together with meaningful demand or visible ranking opportunity.

**Confidence note:** Confidence is directional and based only on the available observed signals.

**What could make the recommendation wrong:** Missing values, incomplete historical coverage, changing search demand, external ranking changes, seasonality, or a trend that does not persist could make a high-ranked page a poor refresh candidate.

The baseline ranking is therefore a prioritization aid, not proof that a content refresh will improve performance.

In [9]:
# ============================================================
# SECTION 3 — TOP-20 REVIEW
# ============================================================

top20 = baseline_output.head(20).copy()

def confidence_note(row):
    score = row["baseline_score"]

    if score >= 100:
        return "High directional confidence; multiple strong observed signals."
    elif score >= 80:
        return "Medium directional confidence; several supporting signals."
    else:
        return "Lower directional confidence; human review needed."

def what_could_make_it_wrong(row):
    problems = []

    if pd.isna(row["trend_pct"]):
        problems.append("missing trend percentage")

    if pd.isna(row["search_volume"]):
        problems.append("missing search demand")

    if pd.isna(row["ctr"]):
        problems.append("missing CTR")

    if pd.isna(row["avg_position"]):
        problems.append("missing position")

    if not problems:
        return (
            "Trend may not persist; seasonality, search-demand changes, "
            "ranking changes, or incomplete historical context could alter the decision."
        )

    return " and ".join(problems) + " could make the recommendation unreliable."

top20["action"] = "human_review_for_possible_refresh"
top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_could_make_it_wrong"] = top20.apply(
    what_could_make_it_wrong,
    axis=1
)

review_columns = [
    "baseline_rank",
    "content_id",
    "baseline_score",
    "baseline_reason_codes",
    "action",
    "confidence_note",
    "what_could_make_it_wrong",
]

top20_review = top20[review_columns].copy()

print("Top-20 review queue:")
display(top20_review)

print("\nTop-20 rows:", len(top20_review))
assert len(top20_review) == 20
print("✓ Exactly 20 recommendations reviewed")

Top-20 review queue:


,baseline_rank,content_id,baseline_score,baseline_reason_codes,action,confidence_note,what_could_make_it_wrong
0,1,content_e8fc703f7ef0,112.700782,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
1,2,content_548f0bf562a9,110.800782,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
2,3,content_b933ddb19b17,109.677934,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
3,4,content_71b773ffcf59,109.341777,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
4,5,content_5b5e85993c2b,109.341777,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
5,6,content_c3ccadb77bc4,108.396767,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
6,7,content_a7d5a0c2535b,108.200271,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
7,8,content_e5a8021fc0d9,108.161278,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
8,9,content_608540486d95,107.999486,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."
9,10,content_9357e2dc8d36,107.763632,declining_with_demand|low_ctr_visible_page|ran...,human_review_for_possible_refresh,High directional confidence; multiple strong o...,"Trend may not persist; seasonality, search-dem..."



Top-20 rows: 20
✓ Exactly 20 recommendations reviewed


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

Some high-ranked pages may still be weak recommendations if the observed decline is based on incomplete data, search demand is very low, or important fields are missing. A high baseline score does not guarantee that a refresh will improve performance.

The baseline does not use `trend_direction` or `trend_pct` as predictive inputs to the score itself, even though these fields are used to identify the observed decline signal. Identifiers such as `content_id` and `client_id` are also excluded from the scoring features.

No product flags, client identifiers, or future outcome fields are used as scoring inputs. The 90-day metrics are treated as historical measurements and should only be used when their measurement window is available before the decision point.

In [10]:
# ============================================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# ============================================================

print("=== POTENTIAL WEAK PICKS ===")

# Pages ranked highly despite incomplete evidence
weak_pick_conditions = (
    baseline_output["search_volume"].isna()
    | baseline_output["ctr"].isna()
    | baseline_output["avg_position"].isna()
    | baseline_output["trend_pct"].isna()
)

weak_picks = baseline_output.loc[
    weak_pick_conditions
].head(10)

print("Potential weak picks in ranked queue:", weak_pick_conditions.sum())

display(weak_picks)

print("\n=== LEAKAGE CHECK ===")

# Fields that must never be scoring inputs
for field in [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
]:
    assert field not in [
        "search_volume",
        "competition",
        "cpc",
        "word_count",
        "char_count",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct",
        "competition_level",
        "content_type",
        "main_intent",
    ]

print("✓ Outcome fields are not scoring inputs")
print("✓ Identifiers are not scoring inputs")

# Confirm no obvious product/future/outcome columns were added
scoring_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "competition_level",
    "content_type",
    "main_intent",
]

suspicious_columns = [
    c for c in scoring_columns
    if any(term in c.lower()
           for term in ["target", "label", "outcome", "product_flag"])
]

print("Suspicious scoring columns:", suspicious_columns)

assert suspicious_columns == []

print("✓ No obvious target/product-flag fields found")
print("\n✓ Weak-pick review and leakage checks complete")

=== POTENTIAL WEAK PICKS ===
Potential weak picks in ranked queue: 4713


,baseline_rank,content_id,client_id,baseline_score,baseline_reason_codes,trend_direction,trend_pct,search_volume,ctr,avg_position,content_age_days,content_type,main_intent
638,639,content_5197b50a6c88,client_4ec9599fc2,95.00,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,NaN,0.0,7.0,374,keyword article,NaN
645,646,content_96688198c848,client_e629fa6598,95.00,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,NaN,0.0,5.4,460,keyword article,transactional
651,652,content_c79aa397ddea,client_9400f1b21c,95.00,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,NaN,0.0,7.0,557,keyword article,NaN
662,663,content_43fb1d5c60f3,client_e629fa6598,95.00,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,NaN,0.0,7.5,502,keyword article,transactional
663,664,content_c799fa5ef1b8,client_e629fa6598,95.00,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,NaN,0.0,7.0,460,keyword article,commercial
680,681,content_14327b4a825b,client_e629fa6598,95.00,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,NaN,0.0,7.3,460,keyword article,commercial
681,682,content_6cdd1cc1ed10,client_4ec9599fc2,95.00,declining_with_demand|low_ctr_visible_page|ran...,down,-100.0,NaN,0.0,6.7,374,keyword article,NaN
1066,1067,content_1e88332a0ee3,client_e629fa6598,93.00,declining_with_demand|low_ctr_visible_page|ran...,down,-90.0,NaN,0.0,7.7,460,keyword article,transactional
1114,1115,content_0f9305fb9daa,client_4e07408562,92.76,declining_with_demand|low_ctr_visible_page|ran...,down,-88.8,NaN,0.0,4.7,504,keyword article,informational
1225,1226,content_5b91e6eac7e3,client_e629fa6598,92.14,declining_with_demand|low_ctr_visible_page|ran...,down,-85.7,NaN,0.0,5.9,460,keyword article,commercial



=== LEAKAGE CHECK ===
✓ Outcome fields are not scoring inputs
✓ Identifiers are not scoring inputs
Suspicious scoring columns: []
✓ No obvious target/product-flag fields found

✓ Weak-pick review and leakage checks complete


## Self-check — VERIFIED

- [x] Every section above is filled — markdown thinking and supporting code are complete.
- [x] The notebook runs top to bottom with no errors — verified using Runtime → Run all.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful words such as observed, measured, directional, and decision-support.
- [x] Baseline action score and ranked queue were generated successfully.
- [x] Top-20 recommendations were reviewed with action, confidence, and failure conditions.
- [x] Weak picks and leakage checks were completed.
- [x] The completed notebook was committed to my repository under `work/notebooks/`.